# Aula 14 - Notebook: Menor Caminho (Dijkstra) e Roteamento Dinâmico

**Projeto:** SCADA-Core Automática — GRUPO 4  
**Unidade:** Planta Automatizada de Seleção e Classificação Óptica de Grãos com Separação Pneumática Tripla  
**Engenharia de Automação e Controle** — ECAA08  

---

### 1. Objetivos do Notebook
Neste notebook implementamos o algoritmo de **Dijkstra** com fila de prioridade (*Min-Heap*):
1. Determinação dos caminhos métricos estritamente mínimos de transporte até os três silos da planta;
2. Simulação de **bloqueios dinâmicos de dutos e válvulas** com peso infinito ($w = \infty$);
3. Recálculo automático de rotas de contingência e detecção formal de nós ilhados.


### 2. Funções de Formatação Tabular ASCII


In [ ]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)


### 3. Estrutura da Rede da Planta e Construtor com Pesos Métricos


In [ ]:
from typing import List, Dict, Any, Tuple, Optional
import heapq

class GrafoTubulacao:
    def __init__(self, vertices: List[str]):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n): self.adj_pesos[i][i] = 0.0
        self.arestas_detalhes: List[Dict[str, Any]] = []

    def adicionar_tubulacao(self, origem: str, destino: str, comprimento_m: float,
                           tag_valvula: str, meio_transporte: str = "Duto", bitola: str = "DN200"):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        self.arestas_detalhes.append({
            "Aresta": f"e{len(self.arestas_detalhes)+1}",
            "Origem": origem,
            "Destino": destino,
            "Comprimento (m)": comprimento_m,
            "Atuador / ISA": tag_valvula,
            "Meio": meio_transporte,
            "Bitola": bitola
        })

    def bloquear_aresta(self, origem: str, destino: str):
        """Simula o fechamento de uma válvula ou bloqueio mecânico na tubulação."""
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 0
        self.adj_pesos[u][v] = float('inf')

    def desbloquear_aresta(self, origem: str, destino: str, peso_original: float):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = peso_original

def criar_rede_padrao() -> GrafoTubulacao:
    nos = [
        "FN-101_Recepcao", "ALIM-101_Calha", "EST-201_Entrada", "BAL-301_Pesagem",
        "VIS-401_Inspecao", "DIV-501_Decisao", "EJE-602_EstacaoB", "EJE-603_EstacaoC",
        "CALHA-602_Secundario", "CALHA-603_Rejeito", "CALHA-701_Terminal",
        "SILO-701_Aprovados", "SILO-702_Secundario", "SILO-703_Rejeito"
    ]
    g = GrafoTubulacao(nos)
    g.adicionar_tubulacao("FN-101_Recepcao", "ALIM-101_Calha", 1.2, "XV-101", "Duto Vertical", "DN200")
    g.adicionar_tubulacao("ALIM-101_Calha", "EST-201_Entrada", 1.8, "c_ALIM", "Calha Vibratória", "L=300mm")
    g.adicionar_tubulacao("EST-201_Entrada", "BAL-301_Pesagem", 2.5, "c_EST/ST-201", "Correia", "L=400mm")
    g.adicionar_tubulacao("BAL-301_Pesagem", "VIS-401_Inspecao", 3.0, "WT-301/FT-301", "Correia", "L=400mm")
    g.adicionar_tubulacao("VIS-401_Inspecao", "DIV-501_Decisao", 1.0, "XS-401/KSA-401", "Zona Óptica", "L=400mm")
    g.adicionar_tubulacao("DIV-501_Decisao", "EJE-602_EstacaoB", 2.0, "ZC-602/p_POS602", "Correia", "L=400mm")
    g.adicionar_tubulacao("EJE-602_EstacaoB", "CALHA-602_Secundario", 0.8, "FY-602/ZSH-602", "Sopro B", "DN15")
    g.adicionar_tubulacao("EJE-602_EstacaoB", "EJE-603_EstacaoC", 1.5, "ST-201", "Correia", "L=400mm")
    g.adicionar_tubulacao("EJE-603_EstacaoC", "CALHA-603_Rejeito", 0.8, "FY-603/ZSH-601", "Sopro C", "DN15")
    g.adicionar_tubulacao("EJE-603_EstacaoC", "CALHA-701_Terminal", 2.2, "Descarga Inércia", "Correia", "L=400mm")
    g.adicionar_tubulacao("CALHA-701_Terminal", "SILO-701_Aprovados", 3.5, "XV-701", "Duto A", "DN250")
    g.adicionar_tubulacao("CALHA-602_Secundario", "SILO-702_Secundario", 4.0, "XV-702", "Duto B", "DN200")
    g.adicionar_tubulacao("CALHA-603_Rejeito", "SILO-703_Rejeito", 3.2, "XV-703", "Duto C", "DN200")
    g.adicionar_tubulacao("SILO-702_Secundario", "FN-101_Recepcao", 12.0, "XV-104", "Elevador Reciclo", "Canecas 6pol")
    return g

rede = criar_rede_padrao()


### 4. Implementação do Motor de Dijkstra: Classe `RoteadorDijkstra`


In [ ]:
class RoteadorDijkstra:
    def __init__(self, grafo: GrafoTubulacao):
        self.g = grafo

    def calcular_menor_caminho(self, origem: str) -> Tuple[Dict[str, float], Dict[str, Optional[str]]]:
        """Executa Dijkstra com Min-Heap a partir do no de origem."""
        dist = {v: float('inf') for v in self.g.vertices}
        pai = {v: None for v in self.g.vertices}
        
        u_origem = origem
        dist[u_origem] = 0.0
        
        # Fila de prioridade: tuplas (distancia, vertice)
        heap = [(0.0, u_origem)]
        
        while heap:
            d_u, u = heapq.heappop(heap)
            
            if d_u > dist[u]:
                continue
                
            u_idx = self.g.v_to_idx[u]
            for v_idx in range(self.g.n):
                peso = self.g.adj_pesos[u_idx][v_idx]
                if peso < float('inf'):
                    v = self.g.idx_to_v[v_idx]
                    if dist[u] + peso < dist[v]:
                        dist[v] = dist[u] + peso
                        pai[v] = u
                        heapq.heappush(heap, (dist[v], v))
                        
        return dist, pai

    def reconstruir_rota(self, origem: str, destino: str, pai: Dict[str, Optional[str]]) -> Optional[List[str]]:
        if origem == destino:
            return [origem]
        if pai[destino] is None:
            return None
        caminho = []
        atual = destino
        while atual is not None:
            caminho.append(atual)
            if atual == origem:
                break
            atual = pai[atual]
        return list(reversed(caminho))


### 5. Execução em Regime Nominal: Rotas Ótimas da Planta


In [ ]:
roteador = RoteadorDijkstra(rede)
dist_nom, pai_nom = roteador.calcular_menor_caminho("FN-101_Recepcao")

dados_nominais = []
for v in rede.vertices:
    rota = roteador.reconstruir_rota("FN-101_Recepcao", v, pai_nom)
    rota_str = " -> ".join([p.split("_")[0] for p in rota]) if rota else "ILHADO / INACESSÍVEL"
    dados_nominais.append({
        "Equipamento": v,
        "Distância Mínima (m)": f"{dist_nom[v]:.1f}" if dist_nom[v] < float('inf') else "INF",
        "Antecessor Ótimo": pai_nom[v] if pai_nom[v] else "---",
        "Rota Otimizada": rota_str
    })

print("=== ROTAS E DISTÂNCIAS MÍNIMAS (DIJKSTRA) EM REGIME NOMINAL ===")
print(formatar_tabela(dados_nominais))


### 6. Simulação de Bloqueio Operacional e Reencaminhamento Dinâmico


In [ ]:
# Cenário de Falha: Válvula XV-701 do Silo A emperrada/bloqueada
print(">>> INJETANDO BLOQUEIO NA ARESTA e11 (CALHA-701 -> SILO-701)...")
rede.bloquear_aresta("CALHA-701_Terminal", "SILO-701_Aprovados")

dist_falha, pai_falha = roteador.calcular_menor_caminho("FN-101_Recepcao")

print(f"Status do SILO-701 após bloqueio:")
print(f"-> Distância calculada: {dist_falha['SILO-701_Aprovados']}")
status_con = 'CONEXO' if dist_falha['SILO-701_Aprovados'] < float('inf') else 'DESCONEXO (ILHADO - ALARME DE INTERTRAVAMENTO)'
print(f'-> Conectividade: {status_con}')

# Restaurando a aresta
rede.desbloquear_aresta("CALHA-701_Terminal", "SILO-701_Aprovados", 3.5)
print(">>> Válvula restaurada com sucesso.")
